# 3장 1강: 교차표와 카이제곱 독립성 검정 이론 — 실습문제

## 실습 목표

- 두 범주형 변수의 교차표를 구성하고 관측빈도를 해석할 수 있다.
- 행 합계와 열 합계를 이용해 기대빈도를 직접 계산할 수 있다.
- 관측빈도와 기대빈도로 카이제곱 통계량과 자유도를 계산할 수 있다.
- 카이제곱 독립성 검정을 수행하고 기대빈도 조건을 확인할 수 있다.
- p-value와 범주별 비율을 함께 사용하여 변수 간 관련성을 해석할 수 있다.

## 실습 환경 / 데이터

- Python
- pandas, NumPy
- scipy.stats
- `ames_housing.csv`

| 컬럼 | 의미 |
|---|---|
| `OverallQual` | 주택의 전반적인 품질 점수 |
| `YearBuilt` | 건축연도 |
| `CentralAir` | 중앙 냉방시설 유무 |
| `KitchenQual` | 주방 품질 |
| `PavedDrive` | 진입로 포장 상태 |

> 모든 검정의 유의수준은 `α = 0.05`입니다.  
> `chi2_contingency()`에서는 강의자료와 동일하게 `correction=False`를 사용합니다.

In [2]:
# 카이제곱 통계량 : 관측빈도가 기대빈도에서 얼마나 벗어났는지를 모든 칸에 걸쳐 합친 숫자
# -> 일반, 재구매를 표로 나타내어 칸으로 나타내면 (30-40)의 제곱 % 40 = 2.5고 나머지 칸도 같은 방식으로 계산하면 약 8.3이다.
# -> x의 제곱 = 모든 칸의 (관측 - 기대)의 제곱 % 기대값

# 카이제곱 독립성 검정 : 두 범주형 변수가 독립이라는 가설을 교차표의 관측빈도로 검토하는 방법
# -> 이용기기(모바일,pc)와 유료회원 가입여부(가입/미가입)과 관련되었는지 검정한다.
# 귀무가설 : 두 변수가 독립적이다.
# 대립가설 : 두 변수는 독립적이지 않다.(연관이 있다.)

## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. Ames Housing 데이터를 `df`에 불러오세요.
3. 데이터 크기, 결측치 수, 컬럼명과 상위 5개 행을 확인하세요.

In [1]:
# 실습 준비 코드를 작성하세요.


# 1. 필요한 라이브러리 불러오기
import pandas as pd
from scipy import stats

# 2. Ames Housing 데이터를 df에 불러오기
df = pd.read_csv('ames_housing.csv')

print(df.isna().sum())

# 3. 데이터의 행과 열 개수, 컬럼명, 상위 5개 행 확인
print("행, 열 개수:", df.shape)
print()
print("컬럼명:", df.columns.tolist())
print()
print("상위 5개 행:")
print(df.head())

SalePrice       0
GrLivArea       0
LotArea         0
OverallQual     0
KitchenQual     0
CentralAir      0
HeatingQC       0
PavedDrive      0
Neighborhood    0
YearBuilt       0
dtype: int64
행, 열 개수: (1460, 10)

컬럼명: ['SalePrice', 'GrLivArea', 'LotArea', 'OverallQual', 'KitchenQual', 'CentralAir', 'HeatingQC', 'PavedDrive', 'Neighborhood', 'YearBuilt']

상위 5개 행:
   SalePrice  GrLivArea  LotArea  OverallQual KitchenQual CentralAir  \
0     208500       1710     8450            7          Gd          Y   
1     181500       1262     9600            6          TA          Y   
2     223500       1786    11250            7          Gd          Y   
3     140000       1717     9550            7          Gd          Y   
4     250000       2198    14260            8          Gd          Y   

  HeatingQC PavedDrive Neighborhood  YearBuilt  
0        Ex          Y      CollgCr       2003  
1        Ex          Y      Veenker       1976  
2        Ex          Y      CollgCr       2001  
3   

---

## 필수 1. 교차표와 카이제곱 통계량 직접 계산

### 문제 1-1. 주택 품질 구간과 중앙 냉방시설의 관계

#### 문제 설명

`OverallQual`을 세 구간으로 나눈 뒤 중앙 냉방시설 유무와의 관계를 확인합니다.

- 낮음: 1~5점
- 보통: 6~7점
- 높음: 8~10점

#### 요구사항

1. `pd.cut()`을 이용해 위 기준으로 `QualityGroup`을 만드세요.
2. 행에는 `QualityGroup`, 열에는 `CentralAir`가 오도록 합계 없는 교차표를 만드세요.
3. `margins=True`인 교차표도 별도로 만들어 행·열 합계를 확인하세요.
4. 합계가 없는 교차표를 NumPy 배열 `observed`로 변환하세요.
5. 행 합계, 열 합계, 전체 합계를 계산하세요.
6. `(행 합계 × 열 합계) / 전체 합계`로 기대빈도를 직접 계산하세요.
7. `Σ(관측-기대)²/기대`로 카이제곱 통계량을 직접 계산하세요.
8. `(행 수-1) × (열 수-1)`로 자유도를 계산하세요.
9. 모든 기대빈도가 5 이상인지 확인하세요.
10. `stats.chi2_contingency(..., correction=False)` 결과와 직접 계산한 값을 비교하세요.
11. p-value를 이용해 두 변수가 관련 있는지 판단하세요.

#### 해석 질문

**Q1.** 교차표 각 칸의 숫자는 무엇을 의미하나요?  
**Q2.** 기대빈도는 어떤 가정 아래 계산되는 값인가요?  
**Q3.** 직접 계산한 카이제곱 통계량과 함수의 결과는 일치하나요?  
**Q4.** 주택 품질 구간과 중앙 냉방시설 유무는 서로 독립이라고 볼 수 있나요?

#### 제출 결과

- 관측빈도 교차표와 주변합
- 기대빈도
- 수동 계산한 카이제곱 통계량과 자유도
- 기대빈도 조건 확인
- 함수 결과와의 비교
- 독립성 판단
- Q1~Q4 답변

In [ ]:
# 필수 1 코드를 작성하세요.

import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("ames_housing.csv")

# 1. QualityGroup 생성
df['QualityGroup'] = pd.cut(
    df['OverallQual'],
    bins=[0, 5, 7, 10],
    labels=['낮음', '보통', '높음']
)

# 2. 합계 없는 교차표 (행: QualityGroup, 열: CentralAir)
crosstab = pd.crosstab(df['QualityGroup'], df['CentralAir'])
print("=== 교차표 (합계 없음) ===")
print(crosstab)

# 3. margins=True인 교차표
crosstab_margins = pd.crosstab(df['QualityGroup'], df['CentralAir'], margins=True)
print("\n=== 교차표 (margins=True) ===")
print(crosstab_margins)

# 4. NumPy 배열로 변환
observed = crosstab.to_numpy()
print("\nobserved (NumPy 배열):")
print(observed)

# 5. 행 합계, 열 합계, 전체 합계
row_totals = observed.sum(axis=1)
col_totals = observed.sum(axis=0)
grand_total = observed.sum()
print(f"\n행 합계: {row_totals}, 열 합계: {col_totals}, 전체 합계: {grand_total}")

# 6. 기대빈도 직접 계산
expected = np.outer(row_totals, col_totals) / grand_total
print("\n기대빈도(직접 계산):")
print(expected)

# 7. 카이제곱 통계량 직접 계산
chi2_manual = ((observed - expected)**2 / expected).sum()
print(f"\n카이제곱 통계량(직접 계산): {chi2_manual:.4f}")

# 8. 자유도 계산
n_rows, n_cols = observed.shape
dof_manual = (n_rows - 1) * (n_cols - 1)
print(f"자유도(직접 계산): {dof_manual}")

# 9. 모든 기대빈도가 5 이상인지 확인
print(f"\n모든 기대빈도 >= 5 여부: {(expected >= 5).all()}")

# 10. scipy와 비교
chi2_scipy, p_value, dof_scipy, expected_scipy = stats.chi2_contingency(observed, correction=False)
print(f"\nscipy 결과: chi2={chi2_scipy:.4f}, dof={dof_scipy}, p-value={p_value:.6f}")
print(f"일치 여부 - chi2: {np.isclose(chi2_manual, chi2_scipy)}, dof: {dof_manual==dof_scipy}, expected: {np.allclose(expected, expected_scipy)}")

# 11. 결론
alpha = 0.05
if p_value < alpha:
    print(f"\np-value({p_value:.6f}) < alpha({alpha}) → 귀무가설 기각")
    print("주택 품질 등급(QualityGroup)과 중앙 냉방시설 유무(CentralAir)는 서로 관련이 있습니다.")
else:
    print(f"\np-value({p_value:.6f}) >= alpha({alpha}) → 귀무가설 기각 실패")
    print("주택 품질 등급과 중앙 냉방시설 유무 사이에 관련이 있다고 보기 어렵습니다.")

=== 교차표 (합계 없음) ===
CentralAir     N    Y
QualityGroup         
낮음            71  467
보통            23  670
높음             1  228

=== 교차표 (margins=True) ===
CentralAir     N     Y   All
QualityGroup                
낮음            71   467   538
보통            23   670   693
높음             1   228   229
All           95  1365  1460

observed (NumPy 배열):
[[ 71 467]
 [ 23 670]
 [  1 228]]

행 합계: [538 693 229], 열 합계: [  95 1365], 전체 합계: 1460

기대빈도(직접 계산):
[[ 35.00684932 502.99315068]
 [ 45.09246575 647.90753425]
 [ 14.90068493 214.09931507]]

카이제곱 통계량(직접 계산): 65.0304
자유도(직접 계산): 2

모든 기대빈도 >= 5 여부: True

scipy 결과: chi2=65.0304, dof=2, p-value=0.000000
일치 여부 - chi2: True, dof: True, expected: True

p-value(0.000000) < alpha(0.05) → 귀무가설 기각
주택 품질 등급(QualityGroup)과 중앙 냉방시설 유무(CentralAir)는 서로 관련이 있습니다.


### 필수 1 답변 작성란

**Q1.** 교차표 각 칸의 숫자는 무엇을 의미하나요?  
-> 두 범주형 변수의 해당 조합에 실제로 속한 주택의 개수인 관측빈도

**Q2.** 기대빈도는 어떤 가정 아래 계산되는 값인가요?  
-> 두 범주형 변수가 서로 독립이라는 귀무가설이 참일 때 각 칸에 기대되는 빈도

**Q3.** 직접 계산한 카이제곱 통계량과 함수의 결과는 일치하나요?  
-> 65.03으로 일치

**Q4.** 주택 품질 구간과 중앙 냉방시설 유무는 서로 독립이라고 볼 수 있나요?
-> 독립적이라고 보기 어려움, p-value가 0.05보다 작아 독립이라는 귀무가설을 기각 (서로 연관이 있음)


---

## 필수 2. 카이제곱 독립성 검정과 비율 해석

### 문제 2-1. 건축연도 구간과 중앙 냉방시설의 관계

#### 문제 설명

건축연도를 다음 세 구간으로 나누고 중앙 냉방시설 설치 여부와 관련이 있는지 확인하세요.

- 1980년 이전
- 1980~1999년
- 2000년 이후

#### 요구사항

1. `pd.cut()`로 `YearBuiltGroup`을 만드세요.
2. `YearBuiltGroup`과 `CentralAir`의 교차표를 만드세요.
3. 다음 가설을 작성하세요.
   - H₀: 건축연도 구간과 중앙 냉방시설 유무는 서로 독립이다.
   - H₁: 건축연도 구간과 중앙 냉방시설 유무는 서로 관련이 있다.
4. 카이제곱 독립성 검정을 수행하세요.
5. 카이제곱 통계량, p-value, 자유도와 기대빈도를 출력하세요.
6. 기대빈도 중 5 미만인 칸의 개수를 확인하세요.
7. `pd.crosstab(..., normalize="index")`로 건축연도 구간별 냉방시설 비율을 계산하세요.
8. 검정 결과와 행 비율을 함께 이용해 관련성을 해석하세요.

#### 해석 질문

**Q1.** 이 문제는 적합도 검정과 독립성 검정 중 무엇을 사용해야 하나요?  
**Q2.** 자유도는 얼마이며 어떻게 계산되나요?  
**Q3.** 기대빈도 조건은 충족되나요?  
**Q4.** 건축연도 구간과 중앙 냉방시설 유무 사이에는 유의한 관련성이 있나요?  
**Q5.** 카이제곱 검정 결과만으로 건축연도가 냉방시설 설치의 원인이라고 결론 내릴 수 있나요?

#### 제출 결과

- 교차표와 가설
- 카이제곱 검정 결과
- 기대빈도 조건
- 행 비율
- 관련성 및 인과관계 해석
- Q1~Q5 답변

In [6]:
# 필수 2 코드를 작성하세요.
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("ames_housing.csv")

# 1. YearBuiltGroup 생성
df['YearBuiltGroup'] = pd.cut(
    df['YearBuilt'],
    bins=[0, 1979, 1999, np.inf],
    labels=['1980년 이전', '1980~1999년', '2000년 이후']
)

# 2. 교차표
crosstab = pd.crosstab(df['YearBuiltGroup'], df['CentralAir'])
print("=== 교차표 ===")
print(crosstab)

# 3. 가설
print("\nH0: 건축연도 구간과 중앙 냉방시설 유무는 서로 독립이다.")
print("H1: 건축연도 구간과 중앙 냉방시설 유무는 서로 관련이 있다.")

# 4, 5. 카이제곱 독립성 검정
chi2, p_value, dof, expected = stats.chi2_contingency(crosstab, correction=False)
print(f"\n카이제곱 통계량: {chi2:.4f}")
print(f"p-value: {p_value:.6f}")
print(f"자유도: {dof}")
print("기대빈도:")
print(expected)

# 6. 기대빈도 5 미만 칸 개수 확인
n_below_5 = (expected < 5).sum()
print(f"\n기대빈도가 5 미만인 칸의 개수: {n_below_5}")

# 7. 행 비율 (건축연도 구간별 냉방시설 비율)
ratio_table = pd.crosstab(df['YearBuiltGroup'], df['CentralAir'], normalize="index")
print("\n=== 건축연도 구간별 냉방시설 비율 ===")
print(ratio_table)

# 8. 해석
alpha = 0.05
if p_value < alpha:
    print(f"\np-value({p_value:.6f}) < alpha({alpha}) → 귀무가설 기각")
    print("건축연도 구간과 중앙 냉방시설 유무는 서로 관련이 있습니다.")
else:
    print(f"\np-value({p_value:.6f}) >= alpha({alpha}) → 귀무가설 기각 실패")
    print("건축연도 구간과 중앙 냉방시설 유무 사이에 관련이 있다고 보기 어렵습니다.")


=== 교차표 ===
CentralAir       N    Y
YearBuiltGroup         
1980년 이전        95  753
1980~1999년       0  224
2000년 이후         0  388

H0: 건축연도 구간과 중앙 냉방시설 유무는 서로 독립이다.
H1: 건축연도 구간과 중앙 냉방시설 유무는 서로 관련이 있다.

카이제곱 통계량: 73.3330
p-value: 0.000000
자유도: 2
기대빈도:
[[ 55.17808219 792.82191781]
 [ 14.57534247 209.42465753]
 [ 25.24657534 362.75342466]]

기대빈도가 5 미만인 칸의 개수: 0

=== 건축연도 구간별 냉방시설 비율 ===
CentralAir             N         Y
YearBuiltGroup                    
1980년 이전        0.112028  0.887972
1980~1999년      0.000000  1.000000
2000년 이후        0.000000  1.000000

p-value(0.000000) < alpha(0.05) → 귀무가설 기각
건축연도 구간과 중앙 냉방시설 유무는 서로 관련이 있습니다.


### 필수 2 답변 작성란

**Q1.** 이 문제는 적합도 검정과 독립성 검정 중 무엇을 사용해야 하나요?  
-> 건축연도 구간과 냉빙시설 유무라는 두 범주형 변수의 관계확인,독립성 검정 사용
**Q2.** 자유도는 얼마이며 어떻게 계산되나요?  
->  3행 2열 교차표 (3-1)*(2-1) = 2
**Q3.** 기대빈도 조건은 충족되나요?  
-> 네 5 미만인 기대빈도가 0칸으로 기대빈도 기준을 만족
**Q4.** 건축연도 구간과 중앙 냉방시설 유무 사이에는 유의한 관련성이 있나요?  
-> *있습니다. p-valur가 0.05보다 작아 두 변수가 독립적이라는 귀무가설을 기각
**Q5.** 카이제곱 검정 결과만으로 건축연도가 냉방시설 설치의 원인이라고 결론 내릴 수 있나요?
-> *없습니다. 카이제곱 독립성 검정은 관련성을 확인할 뿐, 인과관계를 증명하지 못함


---

## 과제. 주방 품질과 진입로 포장 상태의 관계

### 문제 3-1. 두 범주형 변수 재분류 후 독립성 검정

#### 문제 설명

기대빈도가 너무 작은 범주를 줄이기 위해 주방 품질과 진입로 포장 상태를 다음과 같이 재분류합니다.

- `KitchenGroup`
  - 우수: `Ex`, `Gd`
  - 보통 이하: `TA`, `Fa`
- `DriveGroup`
  - 완전 포장: `Y`
  - 미포장·부분포장: `N`, `P`

#### 요구사항

1. 위 기준으로 `KitchenGroup`과 `DriveGroup`을 만드세요.
2. 두 변수의 교차표를 작성하세요.
3. 두 변수가 독립이라는 귀무가설과 관련이 있다는 대립가설을 작성하세요.
4. 카이제곱 독립성 검정을 수행하세요.
5. 카이제곱 통계량, p-value, 자유도와 기대빈도를 출력하세요.
6. 모든 기대빈도가 5 이상인지 확인하세요.
7. 주방 품질 집단별 진입로 포장 비율을 계산하세요.
8. 검정 결과와 비율 차이를 함께 사용하여 두 변수의 관련성을 해석하세요.

#### 해석 질문

**Q1.** 이 과제에서 범주를 재분류한 이유는 무엇인가요?  
**Q2.** 기대빈도 조건은 충족되나요?  
**Q3.** 주방 품질 집단과 진입로 포장 상태는 서로 독립이라고 볼 수 있나요?  
**Q4.** 두 주방 품질 집단의 완전 포장 비율은 각각 얼마인가요?

#### 제출 결과

- 재분류 코드와 교차표
- 가설 설정
- 카이제곱 검정 결과
- 기대빈도 조건 확인
- 행 비율과 결과 해석
- Q1~Q4 답변

In [4]:
# 과제 코드를 작성하세요.
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("ames_housing.csv")

# 1. KitchenGroup, DriveGroup 생성
df['KitchenGroup'] = np.where(df['KitchenQual'].isin(['Ex', 'Gd']), '우수', '보통 이하')
df['DriveGroup'] = np.where(df['PavedDrive'] == 'Y', '완전 포장', '미포장·부분포장')

# 2. 교차표
crosstab = pd.crosstab(df['KitchenGroup'], df['DriveGroup'])
print("=== 교차표 ===")
print(crosstab)

# 3. 가설
print("\nH0: 주방 품질 집단(KitchenGroup)과 진입로 포장 상태(DriveGroup)는 서로 독립이다.")
print("H1: 주방 품질 집단과 진입로 포장 상태는 서로 관련이 있다.")

# 4, 5. 카이제곱 독립성 검정
chi2, p_value, dof, expected = stats.chi2_contingency(crosstab, correction=False)
print(f"\n카이제곱 통계량: {chi2:.4f}")
print(f"p-value: {p_value:.4e}")
print(f"자유도: {dof}")
print("기대빈도:")
print(expected)

# 6. 기대빈도 5 이상 확인
print(f"\n모든 기대빈도 >= 5 여부: {(expected >= 5).all()}")

# 7. 주방 품질 집단별 진입로 포장 비율
ratio_table = pd.crosstab(df['KitchenGroup'], df['DriveGroup'], normalize="index")
print("\n=== 주방 품질 집단별 진입로 포장 비율 ===")
print(ratio_table)

# 8. 해석
alpha = 0.05
if p_value < alpha:
    print(f"\np-value({p_value:.4e}) < alpha({alpha}) → 귀무가설 기각")
    print("주방 품질 집단과 진입로 포장 상태는 서로 관련이 있습니다.")
else:
    print(f"\np-value({p_value:.4e}) >= alpha({alpha}) → 귀무가설 기각 실패")
    print("주방 품질 집단과 진입로 포장 상태 사이에 관련이 있다고 보기 어렵습니다.")


=== 교차표 ===
DriveGroup    미포장·부분포장  완전 포장
KitchenGroup                 
보통 이하              100    674
우수                  20    666

H0: 주방 품질 집단(KitchenGroup)과 진입로 포장 상태(DriveGroup)는 서로 독립이다.
H1: 주방 품질 집단과 진입로 포장 상태는 서로 관련이 있다.

카이제곱 통계량: 48.2523
p-value: 3.7476e-12
자유도: 1
기대빈도:
[[ 63.61643836 710.38356164]
 [ 56.38356164 629.61643836]]

모든 기대빈도 >= 5 여부: True

=== 주방 품질 집단별 진입로 포장 비율 ===
DriveGroup    미포장·부분포장     완전 포장
KitchenGroup                    
보통 이하         0.129199  0.870801
우수            0.029155  0.970845

p-value(3.7476e-12) < alpha(0.05) → 귀무가설 기각
주방 품질 집단과 진입로 포장 상태는 서로 관련이 있습니다.


### 과제 답변 작성란

**Q1.** 이 과제에서 범주를 재분류한 이유는 무엇인가요? 
-> 기대빈도 기준인 5이상을 만족시키기 위해 재분류함

**Q2.** 기대빈도 조건은 충족되나요?  
-> 충족함
-> 모든 기대빈도가 5이상이기 떄문에 True로 뜸

**Q3.** 주방 품질 집단과 진입로 포장 상태는 서로 독립이라고 볼 수 있나요?  
-> 아니오
-> p = 3.7476e-12로 0.05보다 작아 귀무가설을 기각함
-> 주방 품질 등급과 진입로 포장상태는 연관이 있음을 통계적인 근거로 볼 수 있음

**Q4.** 두 주방 품질 집단의 완전 포장 비율은 각각 얼마인가요?'
-> 보통 이하 집단: 완전 포장 비율 87.08%(674/774) / 우수 집단: 완전 포장 비율 97.08%(666/686)


---

## 실습 마무리

1. 교차표에서 관측빈도와 기대빈도는 어떻게 다른가요?
-> 관측빈도 : 데이터에서 실제로 센 개수 / 기대빈도 : 두 변수가 독립이라고 가정할 때 주변합으로 예상되는 개수

2. 기대빈도는 어떤 공식으로 계산하나요?
-> (해당 행 x 해당 열) % 전체 합계

3. 카이제곱 통계량이 커진다는 것은 무엇을 의미하나요?
-> 관측빈도가 독립을 가정한 기대빈도에서 전체적으로 더 크게 벗어난다

4. 독립성 검정과 적합도 검정은 변수 개수와 질문에서 어떻게 다른가요?
-> 독립성 검정 : 두 범주형 변수의 관계를 확인 / 적합도 검정 : 하나의 범주형 변수의 관측분포가 기대한 분포와 일치하는지 확인

5. 기대빈도가 5보다 작은 칸이 있다면 무엇을 고려해야 하나요?
-> 카이제곱 분포로 근사한 p-value의 정확성이 떨어질 수 있다.
